# Nick Full-Data Sampler Check: DDPM500 vs DPM-Solver50

This notebook inspects the one-run sampler comparison requested for `nf_gen_nick_u128_d2p15`.

- DDPM reference label: `raw_train_full` using the standard 500-step DDPM inference.
- DPM test label: `dpm50` using `DPMSolverMultistepScheduler` with `num_steps=50`.
- Run only: full-data Nick-default `u128`, `d2p15`, `dataset_size=32768`.

Run from the repo root on Great Lakes after the DPM50 sample job has completed.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'scripts').exists()), Path.cwd()).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.metrics import batch_power_spectra

RUN_NAME = 'nf_gen_nick_u128_d2p15'
SEED = 123
DDPM_LABEL = 'raw_train_full'
DPM_LABEL = 'dpm50'
MAX_SAMPLES = 512
SWEEP_NAME = 'nf_generalize_nick_data'

SAMPLE_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'samples'
OUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'sampler_compare'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DDPM_PATH = SAMPLE_DIR / f'{RUN_NAME}_seed{SEED}_{DDPM_LABEL}.npz'
DPM_PATH = SAMPLE_DIR / f'{RUN_NAME}_seed{SEED}_{DPM_LABEL}.npz'
CSV_PATH = OUT_DIR / f'nf_generalize_nick_data_{DPM_LABEL}_sampler_compare.csv'
SUMMARY_PATH = OUT_DIR / f'nf_generalize_nick_data_{DPM_LABEL}_sampler_compare_summary.json'

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 11,
})

print('project:', PROJECT_DIR)
print('ddpm:', DDPM_PATH, 'exists=', DDPM_PATH.exists())
print('dpm50:', DPM_PATH, 'exists=', DPM_PATH.exists())
print('output:', OUT_DIR)

## Run The One-Run Comparison Table

This calls the checked-in comparison script restricted to `nf_gen_nick_u128_d2p15`.

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_DIR / 'scripts' / 'compare_nf_generalize_nick_samplers.py'),
    '--project-dir', str(PROJECT_DIR),
    '--run-name', RUN_NAME,
    '--ddpm-label', DDPM_LABEL,
    '--dpm-label', DPM_LABEL,
    '--max-samples', str(MAX_SAMPLES),
    '--output-dir', str(OUT_DIR),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
metrics = pd.read_csv(CSV_PATH)
display_cols = [
    'run_name', 'dataset_tag', 'dataset_size', 'n_compared',
    'hist_l1', 'abs_mean_delta', 'abs_std_delta', 'abs_p01_delta', 'abs_p99_delta',
    'ddpm_pixel_mean', 'dpm_pixel_mean', 'ddpm_pixel_std', 'dpm_pixel_std',
]
display(metrics[[c for c in display_cols if c in metrics.columns]])
if SUMMARY_PATH.exists():
    print(SUMMARY_PATH.read_text())

## Load Samples

In [ ]:
def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None, :, :]
    if arr.ndim != 4 or arr.shape[1] != 1:
        raise ValueError(f'Expected (N,1,H,W) or (N,H,W), got {arr.shape}')
    return arr

def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    if limit is None or len(arr) <= limit:
        return np.array(arr, copy=True)
    idx = np.linspace(0, len(arr) - 1, int(limit), dtype=np.int64)
    return np.array(arr[idx], copy=True)

ddpm = evenly_limit(load_npz_array(DDPM_PATH), MAX_SAMPLES)
dpm = evenly_limit(load_npz_array(DPM_PATH), MAX_SAMPLES)
n = min(len(ddpm), len(dpm))
ddpm = ddpm[:n]
dpm = dpm[:n]
print('ddpm shape:', ddpm.shape, 'finite:', np.isfinite(ddpm).all())
print('dpm50 shape:', dpm.shape, 'finite:', np.isfinite(dpm).all())
print('ddpm range:', float(ddpm.min()), float(ddpm.max()))
print('dpm50 range:', float(dpm.min()), float(dpm.max()))

## Pixel Distribution And Per-Sample Summary

In [ ]:
def sample_summary(arr: np.ndarray, label: str) -> pd.DataFrame:
    flat = arr.reshape(len(arr), -1)
    return pd.DataFrame({
        'sampler': label,
        'sample_mean': flat.mean(axis=1),
        'sample_std': flat.std(axis=1),
        'sample_p01': np.percentile(flat, 1, axis=1),
        'sample_p99': np.percentile(flat, 99, axis=1),
    })

summary_df = pd.concat([
    sample_summary(ddpm, 'DDPM500'),
    sample_summary(dpm, 'DPM-Solver50'),
], ignore_index=True)
display(summary_df.groupby('sampler').agg(['mean', 'std', 'min', 'max']).round(5))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
bins = np.linspace(float(min(ddpm.min(), dpm.min())), float(max(ddpm.max(), dpm.max())), 160)
axes[0].hist(ddpm.ravel(), bins=bins, density=True, histtype='step', lw=2, label='DDPM500')
axes[0].hist(dpm.ravel(), bins=bins, density=True, histtype='step', lw=2, label='DPM-Solver50')
axes[0].set_title('pixel distribution')
axes[0].set_xlabel('normalized field value')
axes[0].set_ylabel('density')
axes[0].legend(frameon=False)

axes[1].hist(summary_df.loc[summary_df.sampler == 'DDPM500', 'sample_mean'], bins=50, alpha=0.55, label='DDPM500')
axes[1].hist(summary_df.loc[summary_df.sampler == 'DPM-Solver50', 'sample_mean'], bins=50, alpha=0.55, label='DPM-Solver50')
axes[1].set_title('per-sample mean')
axes[1].legend(frameon=False)

axes[2].hist(summary_df.loc[summary_df.sampler == 'DDPM500', 'sample_std'], bins=50, alpha=0.55, label='DDPM500')
axes[2].hist(summary_df.loc[summary_df.sampler == 'DPM-Solver50', 'sample_std'], bins=50, alpha=0.55, label='DPM-Solver50')
axes[2].set_title('per-sample std')
axes[2].legend(frameon=False)

fig.tight_layout()
out = OUT_DIR / f'{RUN_NAME}_ddpm500_vs_dpm50_histograms.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()

## Power Spectrum Comparison

In [ ]:
PK_NBINS = 25
pk_ddpm, k = batch_power_spectra(ddpm, nbins=PK_NBINS)
pk_dpm, _ = batch_power_spectra(dpm, nbins=PK_NBINS)
pk_ddpm_mean = np.nanmean(pk_ddpm, axis=0)
pk_dpm_mean = np.nanmean(pk_dpm, axis=0)
ratio = pk_dpm_mean / np.clip(pk_ddpm_mean, 1e-30, None)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
axes[0].plot(k, pk_ddpm_mean, marker='o', label='DDPM500')
axes[0].plot(k, pk_dpm_mean, marker='o', label='DPM-Solver50')
axes[0].set_yscale('log')
axes[0].set_xlabel('k bin')
axes[0].set_ylabel('mean P(k)')
axes[0].set_title('mean power spectrum')
axes[0].legend(frameon=False)

axes[1].axhline(1.0, color='black', lw=1)
axes[1].plot(k, ratio, marker='o', color='tab:purple')
axes[1].set_xlabel('k bin')
axes[1].set_ylabel('DPM50 / DDPM500')
axes[1].set_title('power ratio')
axes[1].grid(alpha=0.25)

fig.tight_layout()
out = OUT_DIR / f'{RUN_NAME}_ddpm500_vs_dpm50_power.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
print('pk ratio low/mid/high:', [float(np.nanmean(x)) for x in np.array_split(ratio[np.isfinite(ratio)], 3)])
plt.show()

## Image Grid

In [ ]:
N_SHOW = 8
idx = np.linspace(0, n - 1, N_SHOW, dtype=int)
vals = np.concatenate([ddpm[idx, 0].ravel(), dpm[idx, 0].ravel()])
vmin, vmax = np.nanpercentile(vals, [1, 99])

fig, axes = plt.subplots(2, N_SHOW, figsize=(2.1 * N_SHOW, 4.4), constrained_layout=True)
for col, i in enumerate(idx):
    axes[0, col].imshow(ddpm[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0, col].set_title(f'DDPM {i}')
    axes[1, col].imshow(dpm[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[1, col].set_title(f'DPM50 {i}')
    for ax in axes[:, col]:
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle(f'{RUN_NAME}: DDPM500 vs DPM-Solver50 samples', y=1.02)
out = OUT_DIR / f'{RUN_NAME}_ddpm500_vs_dpm50_image_grid.png'
fig.savefig(out, bbox_inches='tight')
print('saved', out)
plt.show()

## Files Written

In [ ]:
for path in sorted(OUT_DIR.glob('*dpm50*')):
    print(path, path.stat().st_size)